# Colab Result Analysis (GCN / GAT / GPS)

Phase 5 companion notebook: loads `results/<model>/metrics.json` and `per_class_acc.json`, then builds comparative tables/plots.

In [ ]:
# --- 1) Setup and paths ---
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

# If running inside the project root on Colab, this is already correct.
PROJECT_ROOT = Path('/content/codebase') if Path('/content/codebase').exists() else Path.cwd()
RESULTS_ROOT = PROJECT_ROOT / 'results'
OUTPUT_DIR = RESULTS_ROOT / 'comparisons'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Output dir  : {OUTPUT_DIR}')

In [ ]:
# --- 2) Helpers ---
def load_json(path: Path) -> dict:
    """Load and return JSON dictionary."""
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def model_available(model: str) -> bool:
    """Return True if both metrics and per-class files exist."""
    p = RESULTS_ROOT / model
    return (p / 'metrics.json').exists() and (p / 'per_class_acc.json').exists()


def load_model_artifacts(model: str) -> tuple[dict, dict]:
    """Return (metrics, per_class_acc) for a model."""
    model_dir = RESULTS_ROOT / model
    metrics = load_json(model_dir / 'metrics.json')
    per_class = load_json(model_dir / 'per_class_acc.json')
    return metrics, per_class


def to_float_per_class(per_class: dict) -> dict[int, float]:
    """Convert class-keyed JSON dict to {int: float}."""
    out: dict[int, float] = {}
    for k, v in per_class.items():
        if v is None:
            continue
        out[int(k)] = float(v)
    return out

In [ ]:
# --- 3) Load all available models ---
candidate_models = ['gcn', 'gat', 'gps']
available_models = [m for m in candidate_models if model_available(m)]

if not available_models:
    raise FileNotFoundError('No model artifacts found under results/<model>/')

print('Available models:', available_models)

metrics_by_model: dict[str, dict] = {}
per_class_by_model: dict[str, dict[int, float]] = {}

for model in available_models:
    metrics, per_class = load_model_artifacts(model)
    metrics_by_model[model] = metrics
    per_class_by_model[model] = to_float_per_class(per_class)

print('Loaded metrics for:', ', '.join(metrics_by_model.keys()))

In [ ]:
# --- 4) Overall metrics table ---
rows = []
for model, m in metrics_by_model.items():
    rows.append({
        'model': model.upper(),
        'best_val_acc': float(m.get('best_val_acc', np.nan)),
        'test_acc': float(m.get('test_acc', np.nan)),
        'best_epoch': int(m.get('best_epoch', -1)),
        'epochs_ran': int(m.get('epochs_ran', -1)),
    })

overall_df = pd.DataFrame(rows).sort_values('test_acc', ascending=False).reset_index(drop=True)
display(overall_df)

overall_df.to_csv(OUTPUT_DIR / 'overall_metrics.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'overall_metrics.csv'}")

In [ ]:
# --- 5) Grouped per-class accuracy plot ---
records = []
for model, per_cls in per_class_by_model.items():
    for cls, acc in per_cls.items():
        records.append({'model': model.upper(), 'class_id': int(cls), 'accuracy': float(acc)})

per_class_df = pd.DataFrame(records)
display(per_class_df.head())

plt.figure(figsize=(18, 6))
sns.barplot(data=per_class_df, x='class_id', y='accuracy', hue='model', errorbar=None)
plt.title('Per-Class Accuracy by Model')
plt.xlabel('Class ID')
plt.ylabel('Accuracy')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_class_grouped_accuracy.png', dpi=300)
plt.show()

print(f"Saved: {OUTPUT_DIR / 'per_class_grouped_accuracy.png'}")

In [ ]:
# --- 6) Delta plot (GPS - GCN) if both are available ---
if 'gcn' in per_class_by_model and 'gps' in per_class_by_model:
    gcn = per_class_by_model['gcn']
    gps = per_class_by_model['gps']
    common = sorted(set(gcn.keys()).intersection(gps.keys()))

    delta_df = pd.DataFrame({
        'class_id': common,
        'delta': [gps[c] - gcn[c] for c in common],
    }).sort_values('delta', ascending=False)

    display(delta_df.head(10))

    plt.figure(figsize=(14, 6))
    sns.barplot(data=delta_df, x='class_id', y='delta', color='steelblue', errorbar=None)
    plt.axhline(0.0, color='black', linewidth=1)
    plt.title('Per-Class Accuracy Delta (GPS - GCN)')
    plt.xlabel('Class ID')
    plt.ylabel('Accuracy Delta')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'delta_gps_minus_gcn.png', dpi=300)
    plt.show()

    delta_df.to_csv(OUTPUT_DIR / 'delta_gps_minus_gcn.csv', index=False)
    print(f"Saved: {OUTPUT_DIR / 'delta_gps_minus_gcn.png'}")
    print(f"Saved: {OUTPUT_DIR / 'delta_gps_minus_gcn.csv'}")
else:
    print('Skipping delta plot: requires both results/gcn and results/gps artifacts.')

In [ ]:
# --- 7) Optional: sync comparative outputs to Drive (if mounted) ---
from pathlib import Path
import shutil

drive_out = Path('/content/drive/MyDrive/EEL6878/results/comparisons')
if drive_out.parent.exists():
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUTPUT_DIR, drive_out, dirs_exist_ok=True)
    print(f'Copied {OUTPUT_DIR} -> {drive_out}')
else:
    print('Drive not mounted. Skip this cell or mount Drive first.')